In [1]:
import torch
import torch.nn as nn
from torch import Tensor
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader,SubsetRandomSampler,ConcatDataset
import torch.nn.functional as F

from model import SeqNN

In [ ]:
# Set the default tensor type to float64
# torch.set_default_dtype(torch.float64)
# torch.set_default_dtype(torch.float16)

# # Check the default tensor dtype
# print("Default tensor dtype:", torch.get_default_dtype())

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(device)

cuda:0


In [3]:
# Load the entire model (architecture + weights)
model = torch.load("model.pth")

/tmp/SLURM_195303/ipykernel_1837040/336259025.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("model.pth")


In [4]:
model = model.to(device)

In [ ]:
# model = model.double()
# model = model.to(torch.float16)

In [5]:
# Set the model to evaluation mode (important for inference)
model.eval()

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (re_lu): ReLU()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 96, kernel_size=(11,), stride=(1,), padding=(5,), bias=False)
    (batch_norm): BatchNorm1d(96, eps=0.001, momentum=0.9265, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(96, 96, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(96, eps=0.001, momentum=0.9265, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(96, 96, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(96, eps=0.001, momentum=0.9265, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size

In [ ]:
from torchinfo import summary

summary(model, input_size=(2, 4, 1048576), col_names=["output_size", "num_params"])

In [ ]:
# from pyfaidx import Fasta

In [ ]:
# fasta_file = "/project/fudenber_735/genomes/hg38/hg38.fa"
# genome = Fasta(fasta_file)

# region = "chr12"
# start = 115163136
# end = 116211712

# region = "chr11"
# start = 75429888
# end = 76478464

# region = "chr15"
# start = 63281152
# end = 64329728

In [6]:
# sequence = genome[region][start:end]

sequence = "A" * 1048576

In [7]:
import numpy as np

In [8]:
def one_hot_encode_sequence(sequence_obj):
    # Convert pyfaidx.Sequence object to string
    sequence = str(sequence_obj).upper()

    # Define the mapping from bases to integers
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    # Step 1: Encode the sequence to integers
    encoded_sequence = np.array([base_to_int[base] for base in sequence])

    # Step 2: One-hot encode the sequence
    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    # Step 3: Expand the dimensions to [1, 4, sequence_length]
    input_sequence = np.expand_dims(one_hot_encoded, axis=0)

    return input_sequence

In [9]:
input_sequence = one_hot_encode_sequence(sequence)

In [ ]:
input_sequence.shape

In [ ]:
input_sequence[:,:,0:5]

In [10]:
tf_output = np.load("/scratch1/smaruj/torch_tf_akita/max_pooling1d_2_output.npy")

In [11]:
tf_output = np.transpose(tf_output, (0, 2, 1))

In [ ]:
tf_output.shape

In [12]:
torch.set_printoptions(precision=8)

In [13]:
layer_outputs = {}

# Hook function to capture and store the output of a specific layer
def hook_fn(module, input, output):
    # You can store the output in a dictionary (or any data structure) for later inspection
    layer_outputs[module] = output
    print(f"Output shape of {module}: {output.shape}")
    print(f"Output dtype of {module}: {output.dtype}")
    print(f"Output slice of {module}:\n", output[:, 0:5, 0:10])  # Example slice for inspection

    # Return the output unchanged so the forward pass continues
    return output
    

In [14]:
model.conv_tower.conv_tower[7]

MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)

In [15]:
# after first convolution
# hook_handle = model.re_lu.register_forward_hook(hook_fn)

# after first convolution
# hook_handle = model.conv_block_1.conv.register_forward_hook(hook_fn)

# after first batch norm
# hook_handle = model.conv_block_1.batch_norm.register_forward_hook(hook_fn)

# hook_handle = model.conv_block_1.pool.register_forward_hook(hook_fn)

# end of conv tower
hook_handle = model.conv_tower.conv_tower[7].register_forward_hook(hook_fn)

# end of trunk
# hook_handle = model.conv_reduce.layers[2].register_forward_hook(hook_fn)

# end of residual 2d blocks
# hook_handle = model.residual2d_block6.bn2.register_forward_hook(hook_fn)

# after max pooling
# hook_handle = model.conv_block_1.pool.register_forward_hook(hook_fn)

# after cropping
# hook_handle = model.cropping_2d.register_forward_hook(hook_fn)

In [16]:
# Convert the NumPy array to a PyTorch tensor
input_sequence_tensor = torch.tensor(input_sequence)
# input_sequence_tensor = torch.tensor(input_sequence, dtype=torch.float64)
# input_sequence_tensor = torch.tensor(input_sequence).to(torch.float16)

In [ ]:
input_sequence_tensor.dtype

In [17]:
model.eval()
with torch.no_grad():
    out = model(input_sequence_tensor)

Output shape of MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False): torch.Size([1, 96, 131072])
Output dtype of MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False): torch.float32
Output slice of MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False):
 tensor([[[-0.78984201, -0.39529449, -0.61839682, -0.63700038, -0.63700038,
          -0.63700038, -0.63700038, -0.63700038, -0.63700038, -0.63700038],
         [-0.03348091,  1.13493943,  0.72825688,  0.72741807,  0.72741807,
           0.72741807,  0.72741807,  0.72741807,  0.72741807,  0.72741807],
         [-0.13066970, -0.34338498, -0.27761090, -0.27761090, -0.27761090,
          -0.27761090, -0.27761090, -0.27761090, -0.27761090, -0.27761090],
         [ 0.38393214, -0.19858384, -0.15570728, -0.15570728, -0.15570728,
          -0.15570728, -0.15570728, -0.15570728, -0.15570728, -0.15570728],
         [ 1.45858693,  0.59134179,  0.66303492,  0.66303492,  0.66303492,
  

In [18]:
layer_outputs[model.conv_tower.conv_tower[7]].shape

torch.Size([1, 96, 131072])

In [19]:
output_numpy_array = layer_outputs[model.conv_tower.conv_tower[7]].cpu().numpy()

In [20]:
matrix1_reshaped = output_numpy_array.squeeze(0)
matrix2_reshaped = tf_output.squeeze(0)

In [21]:
import numpy as np

# Assuming matrix1_reshaped and matrix2_reshaped are already shaped (4, 1048576)
# Example: matrix1_reshaped = np.random.rand(4, 1048576), matrix2_reshaped = np.random.rand(4, 1048576)

# Compute Pearson correlation coefficient for each pair of rows
correlations = []
for i in range(matrix1_reshaped.shape[0]):
    row1 = matrix1_reshaped[i]
    row2 = matrix2_reshaped[i]
    
    # Check if either row is constant (i.e., all values are the same, such as all 0s or all 1s)
    if np.all(row1 == row1[0]) and np.all(row2 == row2[0]):
        # If both rows are constant, set correlation to 1 (perfect correlation)
        print(f"Row {i} contains constant values, setting correlation to 1.")
        correlations.append(1.0)
    elif np.all(row1 == 0) or np.all(row2 == 0):
        # If either row is all zeros, the correlation is undefined, so we set NaN
        print(f"Row {i} contains all zeros, skipping correlation.")
        correlations.append(np.nan)
    else:
        # Calculate correlation between each row pair
        corr = np.corrcoef(row1, row2)[0, 1]
        correlations.append(corr)

# Convert to a NumPy array to handle NaN and Inf values
correlations = np.array(correlations)

# Check how many NaNs and Infs exist
print(f"NaN values: {np.isnan(correlations).sum()}")
print(f"Inf values: {np.isinf(correlations).sum()}")

# Remove NaNs and Infs
valid_correlations = correlations[~np.isnan(correlations) & ~np.isinf(correlations)]

# If there are no valid correlations, we can handle it gracefully
if valid_correlations.size > 0:
    # Calculate the average correlation
    average_correlation = np.mean(valid_correlations)
    print("Average correlation:", average_correlation)
else:
    print("No valid correlations found (all are NaN or Inf).")




NaN values: 0
Inf values: 0
Average correlation: 0.9999999999997242


In [ ]:
valid_correlations

In [ ]:
# Remove the hook after usage
hook_handle.remove()

In [ ]:
output_numpy_array = output.cpu().numpy()
    
    matrix1_reshaped = output_numpy_array.squeeze(0)  # Shape becomes (4, 1048576)
    matrix2_reshaped = tf_output.squeeze(0)
    correlation_matrix = np.corrcoef(matrix1_reshaped, matrix2_reshaped)

In [ ]:
# Convert the NumPy array to a PyTorch tensor
# input_sequence_tensor = torch.tensor(input_sequence)

# Now you can move the tensor to the desired device (e.g., GPU or CPU)
x = input_sequence_tensor.to(device)

In [ ]:
x

In [ ]:
model.eval()
with torch.no_grad():
    example_output = model(x, training=False)

In [ ]:
example_output[:,0:5,0:10]

In [ ]:
print(example_output.dtype)

In [ ]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
target0 = example_output[:,0,:]

In [ ]:
target0.shape

In [ ]:
matrix = from_upper_triu(target0, matrix_len=448, num_diags=2)

In [ ]:
np.nanmean(matrix)

In [ ]:
import matplotlib.pyplot as plt

# Plot the matrix
plt.figure(figsize=(8, 8))
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-2, vmax=2)
plt.colorbar()
plt.title('Heatmap of the Matrix')
plt.show()